# Task 4.5 — Experiment on repository sizes

Run from the project root: `src/ifm_konwoledge_enginnering/Package_Manager/Project`

In [1]:
import time
from pathlib import Path
import sys
import pandas as pd

SRC = Path.cwd() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from package_manager.infrastructure.parsing import load_repository, parse_requests_text
from package_manager.use_cases.install import install
from package_manager.domain.verification import verify
from package_manager.infrastructure.solver.knowledge_base import build_knowledge_base


In [2]:
def run_one(N):
    repo_path = BASE / f"repo_{N}.json"
    req_path = BASE / f"requests_{N}.txt"

    t0 = time.perf_counter()
    repo = load_repository(repo_path)
    variables, opt, axioms = build_knowledge_base(repo)
    build_time = time.perf_counter() - t0

    requests = parse_requests_text(req_path.read_text())

    t1 = time.perf_counter()
    try:
        installed = install(repo, requests)
        solve_time = time.perf_counter() - t1
        verified = verify(repo, installed, requests) == []
        return {
            "N": N,
            "versions": sum(len(p.versions) for p in repo.packages.values()),
            "axioms": len(axioms),
            "requests": requests,
            "installed": len(installed),
            "build_time": build_time,
            "solve_time": solve_time,
            "verified": verified,
        }
    except Exception as e:
        solve_time = time.perf_counter() - t1
        return {
            "N": N,
            "versions": sum(len(p.versions) for p in repo.packages.values()),
            "axioms": len(axioms),
            "requests": requests,
            "result": f"no solution: {e}",
            "build_time": build_time,
            "solve_time": solve_time,
        }


## Run Example

In [3]:
BASE = Path('examples')
result = run_one('example')

pd.DataFrame.from_dict(result, orient='index')

,0
N,example
versions,20
axioms,26
requests,"[app==2.0, tool]"
result,no solution: no solution for the given requests
build_time,0.014147
solve_time,0.009844


## Run  Experiment

In [4]:
GROUP = "k12"  # change to your group
BASE = Path("data/repos/repos") / GROUP
Ns = [10, 30, 100, 300, 1000]

results = [run_one(N) for N in Ns]

WARNING Package httpcore=0.14.5 requirement not satisfiable: requirement Requirement(alternatives=[Atom(package='anyio', constraints=[Constraint(op='>=', version=<Version('3')>), Constraint(op='<', version=<Version('4')>)])]) matches no version in the repository
WARNING Package httpcore=0.14.6 requirement not satisfiable: requirement Requirement(alternatives=[Atom(package='anyio', constraints=[Constraint(op='>=', version=<Version('3')>), Constraint(op='<', version=<Version('4')>)])]) matches no version in the repository
WARNING Package httpcore=0.14.7 requirement not satisfiable: requirement Requirement(alternatives=[Atom(package='anyio', constraints=[Constraint(op='>=', version=<Version('3')>), Constraint(op='<', version=<Version('4')>)])]) matches no version in the repository
WARNING Package httpcore=0.15.0 requirement not satisfiable: requirement Requirement(alternatives=[Atom(package='anyio', constraints=[Constraint(op='>=', version=<Version('3')>), Constraint(op='<', version=<Vers

In [5]:
import pandas as pd
pd.DataFrame(results)


,N,versions,axioms,requests,installed,build_time,solve_time,verified,result
0,10,44,22,"[tomli, zipp, comet-sdk]",4.0,0.011127,0.024564,True,NaN
1,30,193,126,"[platformdirs, pyjwt, lumen-api]",9.0,0.043802,0.106320,True,NaN
2,100,1299,1086,"[async-generator, markupsafe-mosaic, umbra-io]",NaN,0.438278,0.623933,NaN,no solution: no solution for the given requests
3,300,6727,6107,"[pyjwt, frozenlist-juniper, nova-core]",4.0,3.240469,5.070201,True,NaN
4,1000,20985,15853,"[annotated-types, h2-alder, sierra-client]",8.0,7.759882,14.751036,True,NaN
